### Import dat z excel souborů stažených z valuo do databáze
[valuo - zakoupené cenové údaje](https://profi.valuo.cz/cadastre/prices/purchased)

In [1]:
# IMPORT DAT - EXCELFILU Z VALUA DO DB A TAHANI GPS Z https://opencagedata.com/dashboard#geocoding
# Zdrojová složka se soubory :  C:\Users\ijttr\OneDrive\Dokumenty\PROG\PYTHON\DATA_ANALYSIS\VALUO\data
#/////////////////////////////////////////////////////////////////////////////////////////

import os
import sys
import pandas as pd
import urllib
import time
import numpy as np
from tqdm import tqdm
from sqlalchemy import create_engine, text
from opencage.geocoder import OpenCageGeocode

# ======= HLAVNÍ PŘEPÍNAČE ======= #
# Zapnutí / vypnutí doplňování GPS souřadnic po importu.
# True  -> po importu Excelů se spustí fetch_missing_coordinates() (API/DB lookup)
# False -> GPS krok se úplně přeskočí, skript jen naimportuje Excel data do DB
RUN_GPS = False

# Ověření, že sloupce z Excelu (po namapování) odpovídají sloupcům v DB tabulce.
# Doporučeno nechat zapnuté - odchytí nesoulad struktury dřív, než se cokoliv zapíše.
VALIDATE_SCHEMA = True

# ======= KONFIGURACE ======= #
API_KEY = "85af71fbd7334627a5b84894066a8a18"
directory = r"E:\\DATA_ANALYSIS\\VALUO\\data"
TABLE_NAME = "Valuo_data"
SCHEMA_NAME = "dbo"

# Zakódování connection stringu
params = urllib.parse.quote_plus(
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost;"
    "Database=VALUO;"
    "Trusted_Connection=yes"
)

# Vytvoření SQLAlchemy engine
connection_url = f"mssql+pyodbc:///?odbc_connect={params}"
engine = create_engine(connection_url)

# Geokódovací API (inicializuje se, jen když je GPS krok zapnutý)
geocoder = OpenCageGeocode(API_KEY) if RUN_GPS else None

# ======= STRUKTURA TABULKY Valuo_data (musí odpovídat skutečné DB!) =======
# Zdroj pravdy: CREATE TABLE [dbo].[Valuo_data] ve VALUO databázi.
# max_length = pro varchar/text sloupce max. počet znaků (None = bez omezení / netextový typ)
# required   = NOT NULL v DB -> musí mít hodnotu, jinak insert spadne
# sql_type   = cílový typ pro konverzi z Excelu ("text", "int", "decimal", "datetime")
TABLE_SCHEMA = {
    "cislo_vkladu":    {"max_length": 50,   "required": True,  "sql_type": "text"},
    "datum_podani":    {"max_length": None, "required": True,  "sql_type": "datetime"},
    "datum_zplatneni": {"max_length": None, "required": True,  "sql_type": "datetime"},
    "listina":         {"max_length": None, "required": True,  "sql_type": "text"},      # text (SQL) - bez omezení délky
    "nemovitost":      {"max_length": 50,   "required": True,  "sql_type": "text"},
    "typ":             {"max_length": 100,  "required": True,  "sql_type": "text"},
    "adresa":          {"max_length": 400,  "required": True,  "sql_type": "text"},
    "cenovy_udaj":     {"max_length": None, "required": True,  "sql_type": "decimal"},
    "mena":            {"max_length": 10,   "required": True,  "sql_type": "text"},
    "plocha":          {"max_length": None, "required": False, "sql_type": "decimal"},
    "typ_plochy":      {"max_length": 300,  "required": False, "sql_type": "text"},
    "popis":           {"max_length": 800,  "required": False, "sql_type": "text"},
    "okres":           {"max_length": 100,  "required": True,  "sql_type": "text"},
    "kat_uzemi":       {"max_length": 100,  "required": True,  "sql_type": "text"},
    "rok":             {"max_length": None, "required": True,  "sql_type": "int"},
    "mesic":           {"max_length": None, "required": True,  "sql_type": "int"},
}

# Výchozí hodnoty pro POVINNÉ (NOT NULL) textové sloupce, pokud v Excelu chybí.
# U povinných číselných/datumových sloupců výchozí hodnotu záměrně nedáváme -
# chybějící cena/datum/rok/měsíc je datová chyba, kterou chceme vidět, ne zamaskovat.
REQUIRED_TEXT_DEFAULTS = {
    "cislo_vkladu": "NEZNÁMÉ",
    "listina": "Neznámá listina",
    "nemovitost": "Neznámé",
    "typ": "Neznámé",
    "adresa": "Neznámá adresa",
    "mena": "CZK",
    "okres": "Neznámý",
    "kat_uzemi": "Neznámé",
}

# ======= FUNKCE ======= #

def get_excel_files(directory):
    """Najde všechny Excel soubory v dané složce a podadresářích."""
    excel_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith((".xls", ".xlsx")):
                excel_files.append(os.path.join(root, file))
    if not excel_files:
        raise FileNotFoundError("❌ Žádné Excel soubory nenalezeny v dané složce!")
    return excel_files


# ======= MAPOVÁNÍ SLOUPCŮ Z EXCELU NA DB ======= #
COLUMN_MAPPING = {
    "Číslo vkladu": "cislo_vkladu",
    "Datum podání": "datum_podani",
    "Datum zplatnění": "datum_zplatneni",
    "Listina": "listina",
    "Nemovitost": "nemovitost",
    "Typ": "typ",
    "Adresa": "adresa",
    "Cenový údaj": "cenovy_udaj",
    "Měna": "mena",
    "Plocha (v m2)": "plocha",
    "Typ plochy": "typ_plochy",
    "Popis": "popis",
    "Okres": "okres",
    "Kat. území": "kat_uzemi",
    "Rok": "rok",
    "Měsíc": "mesic"
}


def read_excel(file):
    """Načte data z Excelu do pandas DataFrame a přejmenuje sloupce podle mapování."""
    df = pd.read_excel(file, dtype=str)
    df.rename(columns=COLUMN_MAPPING, inplace=True)
    return df


def validate_schema_consistency(df):
    """
    Ověří, že sloupce, které insert_to_db() bude zapisovat (= klíče TABLE_SCHEMA),
    skutečně existují jak v namapovaném DataFrame, tak ve skutečné DB tabulce.
    Při jakémkoliv nesouladu skončí s jasnou chybou - raději nic neimportovat
    než importovat do nesedící struktury.
    """
    expected_columns = set(TABLE_SCHEMA.keys())
    df_columns = set(df.columns)

    missing_in_excel = expected_columns - df_columns
    if missing_in_excel:
        raise ValueError(
            f"❌ V Excelu (po namapování) chybí sloupce, které insert_to_db() očekává: "
            f"{sorted(missing_in_excel)}. Zkontroluj COLUMN_MAPPING a hlavičky v Excelu."
        )

    with engine.connect() as conn:
        db_columns_info = conn.execute(text("""
            SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH, IS_NULLABLE
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = :schema AND TABLE_NAME = :table
        """), {"schema": SCHEMA_NAME, "table": TABLE_NAME}).fetchall()

    if not db_columns_info:
        raise ValueError(
            f"❌ Tabulka {SCHEMA_NAME}.{TABLE_NAME} v DB nenalezena (nebo bez sloupců). "
            f"Zkontroluj připojení / název databáze."
        )

    db_columns = {row[0]: row for row in db_columns_info}
    missing_in_db = expected_columns - set(db_columns.keys())
    if missing_in_db:
        raise ValueError(
            f"❌ Sloupce očekávané kódem chybí v DB tabulce {TABLE_NAME}: {sorted(missing_in_db)}. "
            f"Struktura DB neodpovídá TABLE_SCHEMA v kódu."
        )

    # Kontrola max. délky u textových sloupců - pokud kód ořezává na jinou hodnotu,
    # než jakou má DB sloupec, je to buď zbytečné utínání dat, nebo riziko přetečení.
    mismatches = []
    for col, meta in TABLE_SCHEMA.items():
        _, data_type, char_max_len, is_nullable = db_columns[col]
        if meta["max_length"] is not None and data_type in ("varchar", "nvarchar", "char", "nchar"):
            if char_max_len is not None and char_max_len != meta["max_length"]:
                mismatches.append(
                    f"  - {col}: kód ořezává na {meta['max_length']} znaků, DB sloupec má {char_max_len}"
                )
        if not meta["required"] and is_nullable == "NO":
            mismatches.append(
                f"  - {col}: kód počítá se sloupcem jako s volitelným (NULL), ale DB má NOT NULL"
            )

    if mismatches:
        raise ValueError(
            "❌ Nesoulad mezi TABLE_SCHEMA v kódu a skutečnou strukturou DB tabulky:\n"
            + "\n".join(mismatches)
            + "\nOprav TABLE_SCHEMA v kódu nebo strukturu DB, než budeš importovat."
        )

    print(f"✅ Kontrola struktury OK: {len(expected_columns)} sloupců odpovídá DB tabulce {TABLE_NAME}.")


def _to_int_or_none(value):
    """Bezpečně převede hodnotu na int. Vrací None, pokud to nejde."""
    if value is None:
        return None
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return None
    try:
        # Excel občas ukládá čísla jako "2024.0"
        return int(float(value))
    except (ValueError, TypeError):
        return None


def _to_decimal_or_none(value):
    """Bezpečně převede hodnotu na float pro DECIMAL sloupec (podporuje desetinnou čárku i tečku)."""
    if value is None:
        return None
    value = str(value).strip().replace(" ", "").replace("\xa0", "")
    if value == "" or value.lower() == "nan":
        return None
    value = value.replace(",", ".")
    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def _to_datetime_or_none(value):
    """Bezpečně převede hodnotu na datetime pro DATETIME sloupec."""
    if value is None:
        return None
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return None
    parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.to_pydatetime()


def clean_and_convert(df):
    """
    Vyčistí a přetypuje DataFrame podle TABLE_SCHEMA tak, aby přesně odpovídal
    cílovým sloupcům v DB (typ, délka, NOT NULL).
    """
    df = df.replace({np.nan: None, 'nan': None, pd.NA: None})

    for col, meta in TABLE_SCHEMA.items():
        sql_type = meta["sql_type"]

        if sql_type == "int":
            df[col] = df[col].apply(_to_int_or_none)
        elif sql_type == "decimal":
            df[col] = df[col].apply(_to_decimal_or_none)
        elif sql_type == "datetime":
            df[col] = df[col].apply(_to_datetime_or_none)
        else:  # text
            df[col] = df[col].apply(lambda v: None if v is None else str(v))
            if meta["max_length"] is not None:
                df[col] = df[col].str.slice(0, meta["max_length"])

        # Doplnění výchozí hodnoty pro povinné textové sloupce
        if meta["required"] and col in REQUIRED_TEXT_DEFAULTS:
            df[col] = df[col].fillna(REQUIRED_TEXT_DEFAULTS[col])
            df[col] = df[col].apply(lambda v: REQUIRED_TEXT_DEFAULTS[col] if str(v).strip() == "" else v)

    return df


def report_invalid_required_rows(df):
    """
    Najde řádky, kde povinný (NOT NULL) sloupec bez textového defaultu (datum, cena, rok, měsíc)
    vyšel po konverzi jako None - takový řádek by insert do DB stejně odmítl.
    Vrátí (validní_df, počet_odmítnutých_řádků) a odmítnuté řádky vypíše k nahlédnutí.
    """
    required_no_default = [
        col for col, meta in TABLE_SCHEMA.items()
        if meta["required"] and col not in REQUIRED_TEXT_DEFAULTS
    ]

    invalid_mask = df[required_no_default].isnull().any(axis=1)
    invalid_count = int(invalid_mask.sum())

    if invalid_count > 0:
        print(f"⚠️  {invalid_count} řádků má neplatnou/chybějící hodnotu v povinném sloupci "
              f"({', '.join(required_no_default)}) a NEBUDOU importovány:")
        preview_cols = ["cislo_vkladu"] + required_no_default
        preview_cols = [c for c in preview_cols if c in df.columns]
        print(df.loc[invalid_mask, preview_cols].head(20).to_string(index=False))

    return df.loc[~invalid_mask].copy(), invalid_count


def _row_params_bez_nan(row) -> dict:
    """
    Převede řádek na dict parametrů pro SQL dotaz a nahradí VŠECHNY
    chybějící hodnoty (NaN, pd.NA, NaT) skutečným Pythonovým None.

    Nutné navzdory tomu, že _to_decimal_or_none/_to_int_or_none/
    _to_datetime_or_none už vrací None pro chybějící hodnoty: jakmile
    pandas sloupec obsahuje mix None a čísel, automaticky ho POVÝŠÍ na
    dtype float64, ve kterém None nelze reprezentovat - potichu se z něj
    stane numpy.nan. row.to_dict() pak vrátí nan (ne None) a pyodbc to
    odmítne chybou "not a valid instance of data type float", protože
    nan není platná hodnota žádného SQL typu (ani FLOAT, ani DECIMAL).
    pd.isna() spolehlivě pozná NaN/NaT/pd.NA napříč všemi typy sloupců.
    """
    return {
        col: (None if pd.isna(value) else value)
        for col, value in row[list(TABLE_SCHEMA.keys())].to_dict().items()
    }


def insert_to_db(df):
    """Vkládá nové záznamy do databáze s kontrolou duplicit."""
    if df.empty:
        return 0

    new_inserts = 0
    with engine.begin() as conn:
        for _, row in df.iterrows():
            result = conn.execute(text("""
                IF NOT EXISTS (
                    SELECT 1 FROM Valuo_data WHERE 
                    cislo_vkladu = :cislo_vkladu AND datum_podani = :datum_podani AND datum_zplatneni = :datum_zplatneni AND 
                    CAST(listina AS NVARCHAR(MAX)) = :listina AND nemovitost = :nemovitost AND typ = :typ AND adresa = :adresa AND 
                    cenovy_udaj = :cenovy_udaj AND mena = :mena AND ISNULL(plocha, -1) = ISNULL(:plocha, -1) AND 
                    ISNULL(typ_plochy, '') = ISNULL(:typ_plochy, '') AND ISNULL(popis, '') = ISNULL(:popis, '') AND 
                    okres = :okres AND kat_uzemi = :kat_uzemi AND rok = :rok AND mesic = :mesic
                ) 
                BEGIN
                    INSERT INTO Valuo_data (cislo_vkladu, datum_podani, datum_zplatneni, listina, nemovitost, typ, adresa, cenovy_udaj, 
                        mena, plocha, typ_plochy, popis, okres, kat_uzemi, rok, mesic) 
                    VALUES (:cislo_vkladu, :datum_podani, :datum_zplatneni, :listina, :nemovitost, :typ, :adresa, :cenovy_udaj, 
                        :mena, :plocha, :typ_plochy, :popis, :okres, :kat_uzemi, :rok, :mesic);
                END
            """), _row_params_bez_nan(row))

            if result.rowcount > 0:
                new_inserts += 1


    return new_inserts


def fetch_missing_coordinates():
    """Doplňuje GPS souřadnice pro záznamy v databázi, které nemají LAT a LON, s limitem 2500 API dotazů/den."""

    with engine.connect() as conn:
        rows = conn.execute(text("""
            SELECT adresa, LAT, LON FROM Valuo_data WHERE LAT IS NOT NULL AND LON IS NOT NULL
        """)).fetchall()

    existing_coords = {adresa: (LAT, LON) for adresa, LAT, LON in rows}

    with engine.connect() as conn:
        rows = conn.execute(text("""
            SELECT id, adresa FROM Valuo_data WHERE LAT IS NULL AND LON IS NULL AND adresa IS NOT NULL AND adresa <> 'Neznámá adresa'
        """)).fetchall()

    updated_rows = 0
    api_calls = 0
    copied_from_db = 0
    skipped_due_to_api_limit = 0
    api_limit_reached = False

    progress_bar = tqdm(rows, desc="🌍 Stahování GPS souřadnic", unit="záznam", dynamic_ncols=True)

    for row in progress_bar:
        record_id, address = row
        lat, lon, gps_status = None, None, "NULL"

        if address in existing_coords:
            lat, lon = existing_coords[address]
            gps_status = "OK"
            copied_from_db += 1
        else:
            if api_limit_reached:
                skipped_due_to_api_limit += 1
                progress_bar.set_postfix_str(
                    f"📍 DB: {copied_from_db} | 🌍 API: {api_calls} | 🚫 Přeskočeno: {skipped_due_to_api_limit}"
                )
                continue

            time.sleep(1)
            try:
                result = geocoder.geocode(address)
                if result:
                    lat = round(result[0]['geometry']['lat'], 7)
                    lon = round(result[0]['geometry']['lng'], 7)
                    api_calls += 1

                    if 48.55 <= lat <= 51.06 and 12.09 <= lon <= 18.86:
                        existing_coords[address] = (lat, lon)
                        gps_status = "OK"
                    else:
                        lat, lon = None, None
                        gps_status = "ERR"
                else:
                    lat, lon = None, None
                    gps_status = "ERR"

            except Exception as e:
                if "RateLimitExceededError" in str(e):
                    api_limit_reached = True
                    print("\n🚨 DENNÍ LIMIT API (2500 dotazů) BYL DOSAŽEN! 🚨")
                    skipped_due_to_api_limit += 1
                    continue

        with engine.begin() as conn:
            conn.execute(text("""
                UPDATE Valuo_data 
                SET LAT=:lat, LON=:lon, GPS_API_info=:gps_status
                WHERE id=:id
            """), {"lat": lat, "lon": lon, "gps_status": gps_status, "id": record_id})
            updated_rows += 1

        progress_bar.set_postfix_str(
            f"📍 DB: {copied_from_db} | 🌍 API: {api_calls} | 🚫 Přeskočeno: {skipped_due_to_api_limit}"
        )

    return updated_rows, api_calls, copied_from_db


def count_missing_coordinates():
    """Vrátí počet záznamů v DB, které stále nemají GPS souřadnice."""
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT COUNT(*) FROM Valuo_data WHERE LAT IS NULL AND LON IS NULL
        """)).scalar()
    return result


# ======= HLAVNÍ SPUŠTĚNÍ ======= #

if __name__ == "__main__":
    try:
        total_files_processed = 0
        total_records = 0
        newly_inserted_records = 0
        rejected_records = 0

        try:
            excel_files = get_excel_files(directory)
        except FileNotFoundError:
            print("⚠️ Nebyly nalezeny žádné nové soubory ke zpracování.\n")
            excel_files = []

        if excel_files:
            for file in tqdm(excel_files, desc="📂 Zpracování souborů"):
                df = read_excel(file)

                if VALIDATE_SCHEMA:
                    validate_schema_consistency(df)

                total_records += len(df)

                df = clean_and_convert(df)
                df, invalid_count = report_invalid_required_rows(df)
                rejected_records += invalid_count

                new_records = insert_to_db(df)
                newly_inserted_records += new_records
                total_files_processed += 1

        # ======= GPS krok - řízeno přepínačem RUN_GPS na začátku souboru ======= #
        if RUN_GPS:
            updated_rows, api_calls, copied_from_db = fetch_missing_coordinates()
            missing_after = count_missing_coordinates()
        else:
            updated_rows, api_calls, copied_from_db = 0, 0, 0
            missing_after = "GPS krok vypnut (RUN_GPS = False)"
            print("\nℹ️  GPS krok přeskočen (RUN_GPS = False).")

        print(f"\n📊 Statistiky zpracování:")
        print(f"📁 Celkem souborů: {total_files_processed}")
        print(f"📝 Celkem záznamů v Excelech: {total_records}")
        print(f"✅ Nově vložené záznamy: {newly_inserted_records}")
        print(f"⛔ Odmítnuté záznamy (chybí povinná hodnota): {rejected_records}")
        if RUN_GPS:
            print(f"📍 Zkopírované GPS z DB: {copied_from_db}")
            print(f"🌍 Nově stažené GPS přes API: {api_calls}")
            print(f"⏳ Záznamy stále bez souřadnic: {missing_after}")

    except Exception as e:
        print(f"❌ Chyba při zpracování: {e}")

📂 Zpracování souborů:   0%|          | 0/52 [00:00<?, ?it/s]

✅ Kontrola struktury OK: 16 sloupců odpovídá DB tabulce Valuo_data.


C:\Users\ijttr\AppData\Local\Temp\ipykernel_59024\96919587.py:224: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
📂 Zpracování souborů:   2%|▏         | 1/52 [00:01<01:04,  1.27s/it]C:\Users\ijttr\AppData\Local\Temp\ipykernel_59024\96919587.py:224: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")


✅ Kontrola struktury OK: 16 sloupců odpovídá DB tabulce Valuo_data.


📂 Zpracování souborů:   4%|▍         | 2/52 [00:02<01:14,  1.49s/it]C:\Users\ijttr\AppData\Local\Temp\ipykernel_59024\96919587.py:224: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")


...

✅ Kontrola struktury OK: 16 sloupců odpovídá DB tabulce Valuo_data.
✅ Kontrola struktury OK: 16 sloupců odpovídá DB tabulce Valuo_data.


C:\Users\ijttr\AppData\Local\Temp\ipykernel_59024\96919587.py:224: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
📂 Zpracování souborů:  98%|█████████▊| 51/52 [00:30<00:00,  1.68it/s]C:\Users\ijttr\AppData\Local\Temp\ipykernel_59024\96919587.py:224: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")


✅ Kontrola struktury OK: 16 sloupců odpovídá DB tabulce Valuo_data.


📂 Zpracování souborů: 100%|██████████| 52/52 [00:30<00:00,  1.69it/s]


ℹ️  GPS krok přeskočen (RUN_GPS = False).

📊 Statistiky zpracování:
📁 Celkem souborů: 52
📝 Celkem záznamů v Excelech: 2856
✅ Nově vložené záznamy: 2856
⛔ Odmítnuté záznamy (chybí povinná hodnota): 0


In [ ]:
# PRIPOJENI K DB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sqlalchemy import create_engine
import urllib.parse

# Zakódování connection stringu
params = urllib.parse.quote_plus(
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost;"
    "Database=VALUO;"
    "Trusted_Connection=yes"
)

# Vytvoření connection stringu
connection_url = "mssql+pyodbc:///?odbc_connect=%s" % params
engine = create_engine(connection_url)

# Definice SQL dotazu
query = """
select 
/* //////////   cela tabulka   ////////// */ 
     * from [dbo].[Valuo_data] 
     where 1=1
           and okres = 'Hlavní město Praha'
           --and kat_uzemi = 'Bubeneč'
           --and nemovitost = 'budova'
           and cenovy_udaj >200000000
"""

# Načtení dat z databáze pomocí SQL dotazu
try:
    df = pd.read_sql(query, engine)
    # Zobrazení prvních několika řádků načtených dat
    print(df.head())
except Exception as e:
    print(f"Error: {e}")